# Track C (Protein): Continuous Objective vs Categorical CE on Real Proteins

**The ESM-2 analog of synthetic Variant A.** Same backbone, same *discrete* inputs,
identical optimizer / steps / lr / schedule / precision / seed (320). The **only**
thing that changes between conditions is the output objective.

## The one hard part

An oscillator sample has a raw float a discrete token is a rounded version of. A
masked alanine does not. So the continuous condition cannot regress to "the raw
value" — it regresses to a **fixed, multi-dimensional, well-separated continuous
code per token**. Inputs stay discrete (identical to the CE condition); only the
head + loss change. Discrete predictions for evaluation come from
**nearest-prototype lookup**, giving a masked-token recovery accuracy directly
comparable to CE's argmax accuracy.

## Conditions (all share backbone, data, schedule, seed=320)

| Condition | Head | Loss | Decode |
|---|---|---|---|
| **CE** | `Linear(d_model, vocab)` | cross-entropy over 20 AA + specials | argmax |
| **Cont-physchem** | `Linear(d_model, d_code)` | MSE to standardized physicochemical code | nearest-prototype |
| **Cont-random** | `Linear(d_model, d_code)` | MSE to fixed random orthonormal code | nearest-prototype |

`Cont-random` is the control that makes the claim causal: it holds the biological
content fixed (the target is arbitrary) while varying only the **loss geometry**
(continuous regression, no softmax normalization, no hard decision boundary). If
*both* continuous arms reduce distortion relative to CE, the effect comes from the
form of the objective — not from biological information smuggled in via the target.
This directly addresses the CE→MSE confound an ICML reviewer raised.

## What we measure

1. **Geometry** (Shesha harness on intermediate reps): Procrustes distortion **D**,
   RDM similarity, composite stability, under a biological perturbation suite
   (random AA substitution at 1/2/5/10% + sequence reversal). *Prediction*: the
   continuous arms show materially lower **D** and higher RDM similarity than CE.
2. **Task performance on frozen representations** (the clause that decides the review):
   intrinsic masked-token recovery (argmax vs nearest-prototype) **and** a linear
   probe on frozen embeddings for a real downstream task (SCOP structural class /
   remote homology). The bar is **comparable accuracy + better geometry**.
3. **Matched-performance check**: even at checkpoints where CE and Cont reach the
   same recovery accuracy, the geometry gap persists.

## Setup

1. Upload `utils/evaluation_harness.py`, `utils/perturbation_protocol.py`, and
   `utils/bio_continuous_codes.py` to the working directory (or keep this notebook
   under `geometric-alignment-tax/`).
2. GPU runtime. Training the representation-forming layers from scratch under each
   objective is required (a head on a frozen backbone won't work — the distortion
   is baked into the backbone). At SmallBERT ~2–4M scale on 50k proteins this is
   single-digit GPU-hours per condition.

---

In [ ]:
# Install dependencies (Colab)
print("Installing dependencies...")
!pip install -q torch transformers datasets shesha-geometry matplotlib seaborn pandas scipy scikit-learn biopython

import os, sys, gc, time, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, '.')
sys.path.insert(0, './utils')
sys.path.insert(0, '../utils')
sys.path.insert(0, '../../utils')          # geometric-alignment-tax/utils
sys.path.insert(0, '../../../utils')

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration (seed=320 everywhere)

SEED = 320

# --- phase switch ---
PHASE = 'full'   # 'quick' for a fast smoke run, 'full' for the headline result

CONFIG = {
    'quick': dict(n_train=4_000,  n_eval=1_000, epochs=4,  batch_size=64),
    'full':  dict(n_train=50_000, n_eval=2_000, epochs=12, batch_size=64),
}[PHASE]

# --- data / tokenization (matched to synthetic Variant A) ---
SEQ_LEN     = 512                 # sequence length (incl. <cls>/<eos>)
MIN_AA_LEN  = 50                  # min protein length to keep
MAX_AA_LEN  = SEQ_LEN - 2         # truncate body so cls+body+eos <= SEQ_LEN

# --- backbone (~2-4M params; matched across all conditions) ---
D_MODEL   = 256
N_LAYERS  = 4
N_HEADS   = 4
FFN_DIM   = 1024
DROPOUT   = 0.1

# --- continuous code ---
D_CODE    = 8                     # code dimensionality (== #physchem features)

# --- objective / optimization (identical across conditions) ---
MLM_PROB     = 0.15
LR           = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS       = CONFIG['epochs']
BATCH_SIZE   = CONFIG['batch_size']
N_TRAIN      = CONFIG['n_train']
N_EVAL       = CONFIG['n_eval']

# --- perturbation suite ---
SUB_RATES = [0.01, 0.02, 0.05, 0.10]

# --- Shesha harness ---
N_BOOTSTRAP = 5 if PHASE == 'full' else 0
MAX_SAMPLES = 2500

# --- paths ---
OUTPUT_BASE = './results/track_c_protein_continuous/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'
for d in (RESULTS_DIR, CACHE_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

set_seed(SEED)
print(f"Phase: {PHASE.upper()} | device: {DEVICE}")
print(f"Train/eval proteins: {N_TRAIN}/{N_EVAL} | seq_len: {SEQ_LEN} | "
      f"epochs: {EPOCHS} | d_model: {D_MODEL} | d_code: {D_CODE}")

In [ ]:
# Fixed continuous codes + shared tokenization
#
# This is the "one hard part": the continuous condition regresses to a fixed,
# well-separated code per amino acid. We build TWO flavours — physicochemical and
# random-orthonormal — that share the SAME discrete input tokenization as CE.

from bio_continuous_codes import (
    build_protein_codes, nearest_prototype_decode, recovery_accuracy,
    procrustes_distortion, rdm_similarity as np_rdm_similarity, AMINO_ACIDS,
)

CODEBOOKS = {
    'physchem': build_protein_codes(kind='physchem', d=D_CODE, seed=SEED),
    'random':   build_protein_codes(kind='random',   d=D_CODE, seed=SEED),
}

# All conditions share one input vocabulary / tokenization.
TOK2ID     = CODEBOOKS['physchem'].token_to_id
VOCAB_SIZE = CODEBOOKS['physchem'].vocab_size
K_REAL     = len(AMINO_ACIDS)                 # decodable tokens are ids 0..K_REAL-1
MASK_ID = TOK2ID['<mask>']; PAD_ID = TOK2ID['<pad>']
CLS_ID  = TOK2ID['<cls>'];  EOS_ID = TOK2ID['<eos>']; UNK_ID = TOK2ID['<unk>']

# Sanity: both codebooks must agree on the input token ids.
assert CODEBOOKS['physchem'].token_to_id == CODEBOOKS['random'].token_to_id
assert CODEBOOKS['physchem'].vocab_size == CODEBOOKS['random'].vocab_size

def encode_protein(seq, seq_len=SEQ_LEN):
    """Amino-acid string -> fixed-length id array: <cls> body <eos> <pad>..."""
    body = [TOK2ID.get(a, UNK_ID) for a in seq[:MAX_AA_LEN]]
    ids = [CLS_ID] + body + [EOS_ID]
    ids = ids[:seq_len] + [PAD_ID] * max(0, seq_len - len(ids))
    return np.asarray(ids, dtype=np.int64)

print(f"VOCAB_SIZE={VOCAB_SIZE} | decodable amino acids K={K_REAL} | d_code={D_CODE}")
for kind, cb in CODEBOOKS.items():
    print(f"  {kind:9s}: codes {cb.codes.shape} | min prototype distance "
          f"{cb.min_pairwise_distance():.4f} | feats={cb.feature_names}")
print("\nIdentity is recoverable from exact codes:")
for kind, cb in CODEBOOKS.items():
    dec = nearest_prototype_decode(cb.prototypes, cb)
    acc = (dec == cb.prototype_ids).mean()
    print(f"  {kind:9s}: exact-code nearest-prototype accuracy = {acc*100:.1f}%")

In [ ]:
# Load a real protein corpus (UniRef50 / SwissProt subset)
#
# Strategy: HuggingFace datasets (fast) -> stream UniRef50 FASTA -> synthetic
# fallback with natural amino-acid frequencies (so the notebook always runs).

VALID_AAS = set(AMINO_ACIDS)
MAX_SCAN_RECORDS = 400_000

# Natural amino-acid frequencies (UniProt), for the offline fallback only.
_AA_FREQ = {
    'A': 8.25, 'R': 5.53, 'N': 4.06, 'D': 5.46, 'C': 1.38, 'Q': 3.93, 'E': 6.72,
    'G': 7.08, 'H': 2.27, 'I': 5.91, 'L': 9.66, 'K': 5.80, 'M': 2.41, 'F': 3.86,
    'P': 4.74, 'S': 6.61, 'T': 5.36, 'W': 1.10, 'Y': 2.92, 'V': 6.86,
}


def _load_from_huggingface(n, min_len, max_len, seed):
    try:
        from datasets import load_dataset
        print("  trying HuggingFace 'sagawa/uniref50-sample' (streaming)...")
        ds = load_dataset('sagawa/uniref50-sample', split='train', streaming=True)
        seqs, scanned = [], 0
        for rec in ds:
            seq = (rec.get('sequence') or rec.get('text') or '').upper()
            scanned += 1
            if min_len <= len(seq) <= max_len and all(c in VALID_AAS for c in seq):
                seqs.append(seq)
            if len(seqs) >= n or scanned >= MAX_SCAN_RECORDS:
                break
        print(f"  collected {len(seqs)} from HuggingFace (scanned {scanned})")
        return seqs if len(seqs) >= 0.8 * n else None
    except Exception as e:
        print(f"  HuggingFace path failed: {e}")
        return None


def _load_from_fasta(n, min_len, max_len, seed):
    try:
        import urllib.request, gzip
        from Bio import SeqIO
        url = 'https://ftp.uniprot.org/pub/databases/uniprot/uniref/uniref50/uniref50.fasta.gz'
        print("  streaming UniRef50 FASTA (reservoir sampling)...")
        rng = np.random.default_rng(seed)
        seqs, eligible = [], 0
        with urllib.request.urlopen(url, timeout=60) as resp:
            with gzip.GzipFile(fileobj=resp) as fh:
                for i, rec in enumerate(SeqIO.parse(fh, 'fasta')):
                    seq = str(rec.seq).upper()
                    if min_len <= len(seq) <= max_len and all(c in VALID_AAS for c in seq):
                        eligible += 1
                        if len(seqs) < n:
                            seqs.append(seq)
                        else:
                            j = rng.integers(0, eligible)
                            if j < n:
                                seqs[j] = seq
                    if i >= MAX_SCAN_RECORDS or len(seqs) >= n:
                        break
        print(f"  collected {len(seqs)} from FASTA")
        return seqs if len(seqs) >= 0.8 * n else None
    except Exception as e:
        print(f"  FASTA path failed: {e}")
        return None


def _synthetic_proteins(n, min_len, max_len, seed):
    print("  generating synthetic proteins with natural AA frequencies (offline fallback)")
    rng = np.random.default_rng(seed)
    aas = np.array(list(_AA_FREQ.keys()))
    p = np.array(list(_AA_FREQ.values())); p = p / p.sum()
    seqs = []
    for _ in range(n):
        L = int(rng.integers(min_len, max_len + 1))
        seqs.append(''.join(rng.choice(aas, size=L, p=p)))
    return seqs


def load_protein_sequences(n, min_len=MIN_AA_LEN, max_len=MAX_AA_LEN, seed=SEED):
    cache = f"{CACHE_DIR}/proteins_{n}_{min_len}_{max_len}_{seed}.txt"
    if os.path.exists(cache):
        with open(cache) as f:
            seqs = [ln.strip() for ln in f if ln.strip()]
        print(f"Loaded {len(seqs)} cached protein sequences")
        return seqs
    print(f"Loading {n} protein sequences...")
    seqs = (_load_from_huggingface(n, min_len, max_len, seed)
            or _load_from_fasta(n, min_len, max_len, seed)
            or _synthetic_proteins(n, min_len, max_len, seed))
    seqs = seqs[:n]
    with open(cache, 'w') as f:
        f.write('\n'.join(seqs))
    print(f"Cached {len(seqs)} sequences to {cache}")
    return seqs


# Load train + eval, encode to fixed-length id arrays.
_all = load_protein_sequences(N_TRAIN + N_EVAL, seed=SEED)
rng = np.random.default_rng(SEED)
rng.shuffle(_all)
train_seqs, eval_seqs = _all[:N_TRAIN], _all[N_TRAIN:N_TRAIN + N_EVAL]

train_ids = np.stack([encode_protein(s) for s in train_seqs])
eval_ids  = np.stack([encode_protein(s) for s in eval_seqs])
print(f"\ntrain_ids {train_ids.shape} | eval_ids {eval_ids.shape}")
print(f"mean residues/seq (train): {(train_ids < K_REAL).sum(1).mean():.1f}")

In [ ]:
# Perturbation suite (token-id level) + dual-head SmallBERT backbone
#
# Perturbations act on the SAME discrete tokens both conditions see, so the
# geometry comparison isolates the objective. The backbone is identical across
# conditions; only `objective` (and hence head + loss) changes.

def perturb_substitute(ids, rate, rng):
    """Random amino-acid substitution at `rate` of real-residue positions."""
    out = ids.copy()
    for i in range(out.shape[0]):
        pos = np.where(out[i] < K_REAL)[0]
        if len(pos) == 0:
            continue
        n_mut = max(1, int(len(pos) * rate))
        for p in rng.choice(pos, size=n_mut, replace=False):
            alt = int(rng.integers(0, K_REAL))
            while alt == out[i, p]:
                alt = int(rng.integers(0, K_REAL))
            out[i, p] = alt
    return out


def perturb_reverse(ids):
    """Reverse the residue body in place (keep <cls>/<eos>/<pad> positions)."""
    out = ids.copy()
    for i in range(out.shape[0]):
        pos = np.where(out[i] < K_REAL)[0]
        if len(pos) > 1:
            out[i, pos] = out[i, pos[::-1]]
    return out


def build_perturbations(ids, seed=SEED):
    rng = np.random.default_rng(seed)
    pert = {}
    for r in SUB_RATES:
        pert[f"aa_sub_{int(r*100)}pct"] = perturb_substitute(ids, r, rng)
    pert['reverse'] = perturb_reverse(ids)
    return pert


class SmallBERT_Bio(nn.Module):
    """Bidirectional 4-layer Transformer encoder (ESM-2 analog), dual head.

    objective='ce'   -> Linear(d_model, vocab_size), cross-entropy.
    objective='cont' -> Linear(d_model, d_code),     MSE to the fixed code.
    Everything else (embeddings, encoder, norm) is identical across conditions.
    """

    def __init__(self, objective='ce', d_code=D_CODE, vocab_size=VOCAB_SIZE,
                 d_model=D_MODEL, nhead=N_HEADS, num_layers=N_LAYERS,
                 dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN, dropout=DROPOUT):
        super().__init__()
        assert objective in ('ce', 'cont')
        self.objective = objective
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size if objective == 'ce' else d_code)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x, return_hidden=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        pad_mask = (x == PAD_ID)                      # True where padded
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        h = self.encoder(h, src_key_padding_mask=pad_mask)   # bidirectional
        h = self.norm(h)
        out = self.head(h)
        return (out, h) if return_hidden else out


_m = SmallBERT_Bio('ce')
print(f"SmallBERT_Bio (CE):   {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params")
_m = SmallBERT_Bio('cont')
print(f"SmallBERT_Bio (Cont): {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params")
del _m
print("Perturbation suite + dual-head backbone ready")

In [ ]:
# MLM training + recovery + embedding extraction + geometry snapshots
from evaluation_harness import StabilityHarness

harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30,
                           seed=SEED, max_samples=MAX_SAMPLES, n_bootstrap=N_BOOTSTRAP)


def make_mlm_batch(ids):
    """BERT-style dynamic masking (80% mask / 10% random / 10% keep)."""
    inp = ids.clone()
    labels = torch.full_like(ids, -100)
    maskable = ids < K_REAL
    selected = maskable & (torch.rand(ids.shape, device=ids.device) < MLM_PROB)
    labels[selected] = ids[selected]
    r = torch.rand(ids.shape, device=ids.device)
    inp[selected & (r < 0.8)] = MASK_ID
    rand_tok = selected & (r >= 0.8) & (r < 0.9)
    rand_vals = torch.randint(0, K_REAL, ids.shape, device=ids.device)
    inp[rand_tok] = rand_vals[rand_tok]
    return inp, labels, selected


def make_fixed_eval_mask(ids, seed=SEED, mlm_prob=MLM_PROB):
    """Deterministic mask shared by ALL conditions (pure recovery: mask only)."""
    rng = np.random.default_rng(seed)
    inp = ids.copy(); labels = np.full_like(ids, -100); sel = np.zeros(ids.shape, bool)
    for i in range(ids.shape[0]):
        pos = np.where(ids[i] < K_REAL)[0]
        if len(pos) == 0:
            continue
        chosen = rng.choice(pos, size=max(1, int(len(pos) * mlm_prob)), replace=False)
        sel[i, chosen] = True; labels[i, chosen] = ids[i, chosen]; inp[i, chosen] = MASK_ID
    return inp, labels, sel


@torch.no_grad()
def eval_recovery(model, objective, codebook, einp, elab, esel, batch_size=128):
    """Masked-token recovery accuracy: argmax (CE) vs nearest-prototype (Cont)."""
    model.eval(); correct = total = 0
    for i in range(0, len(einp), batch_size):
        x = torch.from_numpy(einp[i:i + batch_size]).long().to(DEVICE)
        out = model(x).cpu().numpy()
        sel = esel[i:i + batch_size]; lab = elab[i:i + batch_size]
        if objective == 'ce':
            pred = out[..., :K_REAL].argmax(-1)
        else:
            pred = nearest_prototype_decode(out, codebook)
        correct += int((pred[sel] == lab[sel]).sum()); total += int(sel.sum())
    return correct / max(total, 1)


@torch.no_grad()
def extract_embeddings(model, ids, batch_size=128):
    """Mean-pooled last-layer hidden states over real-residue positions."""
    model.eval(); outs = []
    for i in range(0, len(ids), batch_size):
        x = torch.from_numpy(ids[i:i + batch_size]).long().to(DEVICE)
        _, h = model(x, return_hidden=True)
        mask = (x < K_REAL).unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        outs.append(pooled.cpu().numpy())
    return np.concatenate(outs, 0)


# Fixed eval masks + a small fixed subset for cheap per-epoch geometry snapshots.
EVAL_INPUT, EVAL_LABELS, EVAL_SEL = make_fixed_eval_mask(eval_ids, seed=SEED)
SNAP_N = min(512, N_EVAL)
SNAP_CLEAN = eval_ids[:SNAP_N]
SNAP_PERT = perturb_substitute(SNAP_CLEAN, 0.05, np.random.default_rng(SEED))
SNAP_INPUT, SNAP_LABELS, SNAP_SEL = make_fixed_eval_mask(SNAP_CLEAN, seed=SEED)


def geometry_snapshot(model, objective, codebook):
    rec = eval_recovery(model, objective, codebook, SNAP_INPUT, SNAP_LABELS, SNAP_SEL)
    Xc = extract_embeddings(model, SNAP_CLEAN)
    Xp = extract_embeddings(model, SNAP_PERT)
    return dict(recovery=rec,
                procrustes_D=procrustes_distortion(Xc, Xp),
                rdm_sim=np_rdm_similarity(Xc, Xp))


def train_condition(objective, codebook=None, epochs=EPOCHS, log=True):
    """Train one condition from scratch. Identical schedule/seed across conditions."""
    set_seed(SEED)
    model = SmallBERT_Bio(objective).to(DEVICE)
    code_tensor = torch.from_numpy(codebook.codes).float().to(DEVICE) if objective == 'cont' else None
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n_batches = (N_TRAIN + BATCH_SIZE - 1) // BATCH_SIZE
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * n_batches)
    Xtrain = torch.from_numpy(train_ids).long()
    order = np.arange(N_TRAIN); history = []
    for ep in range(epochs):
        model.train(); np.random.default_rng(SEED + ep).shuffle(order)
        ep_loss = nb = 0
        for s in range(0, N_TRAIN, BATCH_SIZE):
            batch = Xtrain[order[s:s + BATCH_SIZE]].to(DEVICE)
            inp, labels, sel = make_mlm_batch(batch)
            if sel.sum() == 0:
                continue
            out = model(inp)
            if objective == 'ce':
                loss = F.cross_entropy(out[sel], labels[sel])
            else:
                loss = F.mse_loss(out[sel], code_tensor[labels[sel]])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            ep_loss += loss.item(); nb += 1
        snap = geometry_snapshot(model, objective, codebook)
        snap.update(epoch=ep + 1, train_loss=ep_loss / max(nb, 1))
        history.append(snap)
        if log:
            print(f"    ep {ep+1:2d}/{epochs} loss={snap['train_loss']:.4f} "
                  f"recovery={snap['recovery']*100:5.1f}%  D={snap['procrustes_D']:.4f}  "
                  f"rdm_sim={snap['rdm_sim']:.4f}")
    return model, history


print("Training / recovery / embedding / snapshot helpers ready")

In [ ]:
# Run all conditions: train -> recovery -> geometry (Shesha + Procrustes D)
import pandas as pd

CONDITIONS = [
    ('CE',            'ce',   None),
    ('Cont-physchem', 'cont', 'physchem'),
    ('Cont-random',   'cont', 'random'),
]

perturbed_ids = build_perturbations(eval_ids, seed=SEED)
trained, histories, rows, clean_emb = {}, {}, [], {}

for name, obj, cbkey in CONDITIONS:
    print('=' * 70); print(f"CONDITION: {name}"); print('=' * 70)
    cb = CODEBOOKS[cbkey] if cbkey else None
    model, hist = train_condition(obj, cb)
    trained[name], histories[name] = model, hist

    rec = eval_recovery(model, obj, cb, EVAL_INPUT, EVAL_LABELS, EVAL_SEL)
    Xc = extract_embeddings(model, eval_ids); clean_emb[name] = Xc
    print(f"  full-eval masked-token recovery: {rec*100:.2f}%")

    for pname, pids in perturbed_ids.items():
        Xp = extract_embeddings(model, pids)
        res = harness.evaluate(model_name=name, embeddings_clean=Xc,
                               embeddings_perturbed=Xp, perturbation_name=pname)
        D = procrustes_distortion(Xc, Xp)
        rows.append(dict(condition=name, objective=obj, code=cbkey or 'none',
                         perturbation=pname, recovery_acc=rec, procrustes_D=D,
                         rdm_similarity=res.rdm_similarity_score,
                         composite_stability=res.composite_stability,
                         pert_stability=res.perturbation_stability_score))
        print(f"    {pname:14s}  D={D:.4f}  rdm_sim={res.rdm_similarity_score:.4f}  "
              f"composite={res.composite_stability:.4f}")
    cleanup_gpu()

df = pd.DataFrame(rows)
df.to_csv(f"{RESULTS_DIR}/protein_geometry_detailed.csv", index=False)

agg = (df.groupby(['condition', 'code'])
         .agg(recovery_acc=('recovery_acc', 'first'),
              mean_procrustes_D=('procrustes_D', 'mean'),
              mean_rdm_similarity=('rdm_similarity', 'mean'),
              mean_composite=('composite_stability', 'mean'))
         .reset_index())
agg.to_csv(f"{RESULTS_DIR}/protein_geometry_summary.csv", index=False)
print('\n' + '=' * 70); print("GEOMETRY SUMMARY (lower D = more stable; higher rdm_sim = better)")
print('=' * 70)
print(agg.to_string(index=False))

In [ ]:
# Frozen linear probe on a real downstream task
#
# The clause that decides the review: a linear probe on FROZEN embeddings (the
# probe cannot reshape the geometry). Real task = SCOP structural class from the
# TAPE remote-homology benchmark. Bar to clear: Cont retains COMPARABLE accuracy
# to CE while showing the geometry improvement.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, accuracy_score


def load_downstream_protein(label_col='class_label', max_train=4000, max_test=1500):
    """Returns (train_seqs, ytr, test_seqs, yte, name) or None."""
    try:
        from datasets import load_dataset
        ds = load_dataset('proteinea/remote_homology')
        tr_split = ds['train']
        te_split = ds['test'] if 'test' in ds else ds['validation']

        def grab(split, m):
            seqs, labs = [], []
            for r in split:
                s = (r.get('primary') or r.get('sequence') or '').upper()
                s = ''.join(c for c in s if c in VALID_AAS)
                if len(s) >= 30 and label_col in r and r[label_col] is not None:
                    seqs.append(s); labs.append(int(r[label_col]))
                if len(seqs) >= m:
                    break
            return seqs, np.array(labs)

        Xtr, ytr = grab(tr_split, max_train)
        Xte, yte = grab(te_split, max_test)
        # Keep only classes present in train (fold-holdout test can differ).
        keep = np.isin(yte, np.unique(ytr))
        Xte = [s for s, k in zip(Xte, keep) if k]; yte = yte[keep]
        return Xtr, ytr, Xte, yte, f"proteinea/remote_homology[{label_col}]"
    except Exception as e:
        print(f"  downstream load failed ({e}); probe skipped.")
        return None


def probe_condition(name, Xtr_seq, ytr, Xte_seq, yte):
    model = trained[name]
    Etr = extract_embeddings(model, np.stack([encode_protein(s) for s in Xtr_seq]))
    Ete = extract_embeddings(model, np.stack([encode_protein(s) for s in Xte_seq]))
    sc = StandardScaler().fit(Etr)
    clf = LogisticRegression(max_iter=3000, C=1.0)
    clf.fit(sc.transform(Etr), ytr)
    pred = clf.predict(sc.transform(Ete))
    return balanced_accuracy_score(yte, pred), accuracy_score(yte, pred)


probe_rows = []
_task = load_downstream_protein()
if _task is not None:
    Xtr_seq, ytr, Xte_seq, yte, task_name = _task
    print(f"Downstream task: {task_name} | classes={len(np.unique(ytr))} | "
          f"train={len(ytr)} test={len(yte)}")
    for name, _, _ in CONDITIONS:
        bal, acc = probe_condition(name, Xtr_seq, ytr, Xte_seq, yte)
        probe_rows.append(dict(condition=name, probe_balanced_acc=bal, probe_acc=acc))
        print(f"  {name:14s}  balanced_acc={bal*100:5.2f}%  acc={acc*100:5.2f}%")
    probe_df = pd.DataFrame(probe_rows)
    probe_df.to_csv(f"{RESULTS_DIR}/protein_probe.csv", index=False)
    print('\n' + probe_df.to_string(index=False))
    print("\nBar: Cont arms within a few points of CE on the probe = comparable "
          "performance. Combined with lower D, that is the headline result.")
else:
    probe_df = pd.DataFrame(columns=['condition', 'probe_balanced_acc', 'probe_acc'])

In [ ]:
# Matched-performance check
#
# Confound: CE and MSE have different loss scales/landscapes, so a skeptic can
# argue the geometry gap is an optimization artifact. Rebuttal: even at
# checkpoints where CE and Cont reach the SAME recovery accuracy, the geometry
# gap (D) persists. We use the per-epoch (recovery, D) trajectory of each arm and
# compare D at matched recovery levels by interpolation.

import matplotlib.pyplot as plt

mh = pd.DataFrame([dict(condition=name, **h) for name, hist in histories.items() for h in hist])
mh.to_csv(f"{RESULTS_DIR}/protein_matched_history.csv", index=False)


def interp_D_at(hist, levels):
    h = sorted(hist, key=lambda d: d['recovery'])
    recs = np.array([d['recovery'] for d in h]); Ds = np.array([d['procrustes_D'] for d in h])
    return np.interp(levels, recs, Ds)


ce_hist = histories['CE']
ce_recs = [d['recovery'] for d in ce_hist]
print("Procrustes D at MATCHED recovery accuracy (CE vs Cont):\n")
matched_rows = []
for cont in ['Cont-physchem', 'Cont-random']:
    ch = histories[cont]; c_recs = [d['recovery'] for d in ch]
    lo, hi = max(min(ce_recs), min(c_recs)), min(max(ce_recs), max(c_recs))
    if hi <= lo:
        print(f"  {cont}: no recovery-accuracy overlap with CE (cannot match).")
        continue
    levels = np.linspace(lo, hi, 5)
    Dce, Dcont = interp_D_at(ce_hist, levels), interp_D_at(ch, levels)
    print(f"  {cont} vs CE  (overlap recovery {lo*100:.1f}-{hi*100:.1f}%):")
    for lv, dce, dco in zip(levels, Dce, Dcont):
        print(f"    recovery={lv*100:5.1f}%   D_CE={dce:.4f}   D_{cont}={dco:.4f}   "
              f"gap={dce-dco:+.4f}")
        matched_rows.append(dict(cont=cont, recovery=lv, D_CE=dce, D_cont=dco, gap=dce - dco))
if matched_rows:
    pd.DataFrame(matched_rows).to_csv(f"{RESULTS_DIR}/protein_matched_D.csv", index=False)

# Plot: D vs recovery trajectory per condition.
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'CE': '#DC2626', 'Cont-physchem': '#2563EB', 'Cont-random': '#16A34A'}
for name, hist in histories.items():
    recs = [d['recovery'] * 100 for d in hist]; Ds = [d['procrustes_D'] for d in hist]
    ax.plot(recs, Ds, 'o-', color=colors.get(name), label=name, linewidth=2, markersize=6)
    for d in hist[::max(1, len(hist)//4)]:
        ax.annotate(f"ep{d['epoch']}", (d['recovery']*100, d['procrustes_D']),
                    fontsize=7, alpha=0.6)
ax.set_xlabel('Masked-token recovery accuracy (%)'); ax.set_ylabel('Procrustes distortion D (5% sub)')
ax.set_title('Matched-performance check: geometry gap persists at equal recovery', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/protein_matched_performance.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Headline figure: comparable accuracy + better geometry
import matplotlib.pyplot as plt

colors = {'CE': '#DC2626', 'Cont-physchem': '#2563EB', 'Cont-random': '#16A34A'}
sub_df = df[df['perturbation'].str.startswith('aa_sub')].copy()
sub_df['rate'] = sub_df['perturbation'].str.extract(r'(\d+)').astype(int)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# A: Procrustes D vs substitution rate (lower = better)
ax = axes[0]
for name in colors:
    sub = sub_df[sub_df['condition'] == name].sort_values('rate')
    ax.plot(sub['rate'], sub['procrustes_D'], 'o-', color=colors[name], label=name, lw=2)
ax.set_xlabel('AA substitution rate (%)'); ax.set_ylabel('Procrustes distortion D')
ax.set_title('A. Geometric distortion (lower = better)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

# B: RDM similarity vs substitution rate (higher = better)
ax = axes[1]
for name in colors:
    sub = sub_df[sub_df['condition'] == name].sort_values('rate')
    ax.plot(sub['rate'], sub['rdm_similarity'], 'o-', color=colors[name], label=name, lw=2)
ax.set_xlabel('AA substitution rate (%)'); ax.set_ylabel('RDM similarity')
ax.set_title('B. Relational geometry (higher = better)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

# C: recovery + probe accuracy (task performance must be comparable)
ax = axes[2]
names = list(colors); x = np.arange(len(names)); w = 0.38
rec = [df[df['condition'] == n]['recovery_acc'].iloc[0] * 100 for n in names]
if len(probe_df):
    prb = [float(probe_df[probe_df['condition'] == n]['probe_balanced_acc'].iloc[0]) * 100
           if (probe_df['condition'] == n).any() else np.nan for n in names]
else:
    prb = [np.nan] * len(names)
ax.bar(x - w/2, rec, w, label='masked recovery', color=[colors[n] for n in names], alpha=0.9)
ax.bar(x + w/2, prb, w, label='probe balanced acc', color=[colors[n] for n in names], alpha=0.5, hatch='//')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('accuracy (%)'); ax.set_title('C. Task performance (must be comparable)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# D: mean composite stability
ax = axes[3]
comp = [df[df['condition'] == n]['composite_stability'].mean() for n in names]
ax.bar(x, comp, color=[colors[n] for n in names])
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('mean composite stability'); ax.set_title('D. Composite stability (higher = better)', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

fig.suptitle('Track C (Protein): continuous objective reduces geometric distortion '
             'while retaining task performance', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/protein_headline.png", dpi=200, bbox_inches='tight')
plt.show()

# One-line verdict
ce_D = df[df['condition'] == 'CE']['procrustes_D'].mean()
print('\nVERDICT (mean Procrustes D across perturbations):')
for n in names:
    d = df[df['condition'] == n]['procrustes_D'].mean()
    print(f"  {n:14s}: D={d:.4f}" + (f"  ({ce_D/d:.2f}x lower distortion than CE)" if n != 'CE' and d > 0 else ""))

## What this buys

If the result lands as predicted, the sentence the reviewer says is missing
becomes supported **on real proteins, under matched conditions**:

> A continuous regression objective (MSE to a fixed per-token code) reduces
> geometric distortion **D** by a measured factor relative to categorical
> cross-entropy, while retaining comparable masked-token recovery and comparable
> frozen-probe downstream accuracy.

The two confounds are closed from both directions:

- **Random-orthonormal control** (`Cont-random`) holds the biological content of
  the target fixed (it is arbitrary) and varies only the loss geometry. If it
  *also* beats CE on geometry, the effect is the objective form — not biology
  smuggled in through the physicochemical target. This is the clean answer to
  "MSE-to-prototypes is just relabeled CE."
- **Matched-performance check** shows the geometry gap persists at checkpoints
  where CE and Cont reach equal recovery accuracy, so the gap is not an
  optimization-scale artifact.

### Optional strengtheners
- **Scaling:** rerun with `D_MODEL`/`N_LAYERS` bumped (e.g. 2-3 sizes) and show
  the CE geometry gap does **not** close with scale — echoing the ESM-2
  "scaling is illusory" headline inside this controlled setting.
- **More physchem dims:** raise `D_CODE` and add features (flexibility, SASA,
  pKa) to confirm robustness of the physicochemical arm.

### Outputs (under `results/track_c_protein_continuous/results/`)
- `protein_geometry_detailed.csv`, `protein_geometry_summary.csv`
- `protein_probe.csv`
- `protein_matched_history.csv`, `protein_matched_D.csv`
- `protein_headline.png`, `protein_matched_performance.png`

The DNA track (`Track_C_DNA_Continuous.ipynb`) mirrors this exactly with
`SmallStripedHyena` (the Evo 2 analog), human chr22, and adds the
**reverse-complement** perturbation that ties back to the Evo 2 texture finding.